# ASA-Transformer vs Dense Transformer — GPU Benchmark (Kaggle T4 x2)

**Adaptive Selective Attention (ASA)** — Buenahora Ormaza (2026)  
Benchmark escalado: **1024 tokens**, **~500K params**, GPU T4.  

### Optimizaciones activas en esta versión
| # | Optimización | Impacto |
|---|---|---|
| 1 | Pass-1 bajo `torch.no_grad()` | Libera matriz O(N²) del grafo → −70% VRAM training |
| 2 | Advanced-index gather `k[bh, sel, :]` | Índice O(N·A) en vez de O(N·A·D) → −32x tensor de índice |
| 3 | Pass-2 via SDPA reshape `(B·H·N, 1, A)` | FlashAttention-2 en sparse attention |
| 4 | Margin loss con `enable_grad` local | Mantiene señal de routing sin coste global |

In [ ]:
# ── Cell 1: Environment ──────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pymbbo'])

import math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, List

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  VRAM={p.total_memory/1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active device: {DEVICE}')

# SDPA / FlashAttention-2 check
if torch.cuda.is_available() and hasattr(F, 'scaled_dot_product_attention'):
    _q = torch.randn(2, 4, 64, 32, device='cuda')
    _ = F.scaled_dot_product_attention(_q, _q, _q, is_causal=True)
    del _q; torch.cuda.empty_cache()
    print('[OK] FlashAttention-2 (SDPA) active')

In [ ]:
# ── Cell 2: Gather + Pass-2 helpers ──────────────────────────────────

def _gather_kv(
    k: torch.Tensor,   # (B, H, N_total, D)
    v: torch.Tensor,   # (B, H, N_total, D)
    sel: torch.Tensor, # (B, N, A)  integer selection indices
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Memory-efficient gather via advanced indexing.

    OLD (broken): k.unsqueeze(2).expand(B, H, N, N_total, D)  → O(N·N_total·D) index ~1 GB
    OLD (fix 1):  flat_idx = (sel * D + d_off)                → O(N·A·D) index ~270 MB
    NEW (fix 2):  k_2d[bh_idx, sel_bh, :]                    → O(N·A) index ~8 MB  ✓
    """
    B, H, N_total, D = k.shape
    N, A = sel.shape[1], sel.shape[2]

    k_2d = k.reshape(B * H, N_total, D)              # view, no copy
    v_2d = v.reshape(B * H, N_total, D)              # view, no copy

    # (B, N, A) → (B*H, N*A)  — view only
    sel_bh = sel.unsqueeze(1).expand(B, H, N, A).reshape(B * H, N * A)

    # Advanced indexing: no D-expanded index tensor
    bh_idx = torch.arange(B * H, device=k.device).unsqueeze(1)  # (B*H, 1)
    k_gath = k_2d[bh_idx, sel_bh].reshape(B, H, N, A, D)
    v_gath = v_2d[bh_idx, sel_bh].reshape(B, H, N, A, D)
    return k_gath, v_gath


def _pass2_attention(
    q:      torch.Tensor,   # (B, H, N, D)
    k_gath: torch.Tensor,   # (B, H, N, A, D)
    v_gath: torch.Tensor,   # (B, H, N, A, D)
    scale:  float,
) -> torch.Tensor:          # (B, H, N, D)
    """
    Pass-2 sparse attention.
    On CUDA: reshape to (B·H·N, 1, A) → FlashAttention-2 (SDPA).
    CPU:     vectorised element-wise multiply + reduce.
    """
    B, H, N, A, D = k_gath.shape
    if hasattr(F, 'scaled_dot_product_attention') and q.is_cuda:
        # Each query position is an independent flash-attention batch
        q_s = q.reshape(B * H * N, 1, D)
        k_s = k_gath.reshape(B * H * N, A, D)
        v_s = v_gath.reshape(B * H * N, A, D)
        return F.scaled_dot_product_attention(q_s, k_s, v_s).reshape(B, H, N, D)
    # CPU / no-SDPA fallback
    scores  = (q.unsqueeze(-2) * k_gath).sum(-1) * scale
    return (F.softmax(scores, dim=-1).unsqueeze(-1) * v_gath).sum(-2)


print('[OK] Gather + Pass-2 helpers defined.')

In [ ]:
# ── Cell 3: Core ASA Attention Module ────────────────────────────────

class AdaptiveSelectiveAttention(nn.Module):
    """
    ASA: Score-gated Two-Pass Causal Attention with Selection Sharing.

    Key design decisions for speed + memory + quality:

    1. Pass-1 routing under torch.no_grad()
       Selection indices are integers — non-differentiable by definition.
       Running routing without grad frees the O(N²) matrix after topk:
       no backward graph, immediate deallocation, ~70% VRAM saving.

    2. Advanced-index gather
       Index tensor is (B·H, N·A) int64 — ~8 MB for A=128.
       Previous flat_idx was (B·H, N·A·D) — ~1 GB for A=128.

    3. Pass-2 via SDPA (FlashAttention-2)
       Reshape to (B·H·N, 1, A): each query position is a flash batch.
       Zero extra memory allocation, GPU-fused QK^T+softmax+@V.

    4. Margin loss via local enable_grad
       Recomputes imp with grad only when explicitly requested.
       No cost to the common case (return_aux_loss=False).
    """

    def __init__(self, d_model: int, nhead: int, max_a: int = 64,
                 is_router: bool = True, dropout: float = 0.0,
                 margin_delta: float = 0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.d_model      = d_model
        self.nhead        = nhead
        self.head_dim     = d_model // nhead
        self.max_a        = max_a
        self.is_router    = is_router
        self.scale        = 1.0 / math.sqrt(self.head_dim)
        self.margin_delta = margin_delta
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model, d_model)
        self.v_proj   = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, x, selection_indices=None, max_a=None,
                return_aux_loss=False, kv_cache=None):
        B, N, _ = x.shape
        H, D    = self.nhead, self.head_dim

        def _proj(lin): return lin(x).view(B, N, H, D).transpose(1, 2)
        q = _proj(self.q_proj)
        k = _proj(self.k_proj)
        v = _proj(self.v_proj)

        if kv_cache is not None:
            ck, cv = kv_cache
            k = torch.cat([ck, k], dim=2)
            v = torch.cat([cv, v], dim=2)
        new_kv  = (k, v)
        N_total = k.shape[2]
        budget  = max_a if max_a is not None else self.max_a

        # ── Full attention fallback (FlashAttention-2) ───────────────
        if budget >= N_total:
            if hasattr(F, 'scaled_dot_product_attention') and x.is_cuda:
                dp  = self.dropout.p if self.training else 0.0
                out = F.scaled_dot_product_attention(
                    q, k, v, is_causal=(N == N_total), dropout_p=dp)
            else:
                s = (q @ k.transpose(-2, -1)) * self.scale
                if N == N_total:
                    s = s + torch.triu(torch.full((N,N), float('-inf'), device=x.device), 1)[None,None]
                out = F.softmax(s, dim=-1) @ v
            out = self.out_proj(out.transpose(1,2).contiguous().view(B, N, self.d_model))
            fi  = torch.arange(N_total, device=x.device).view(1,1,-1).expand(B, N, -1)
            return out, fi, None, new_kv

        aux_loss = None

        # ── Pass-1 routing: NO GRAD (selection is non-differentiable) ─
        if self.is_router or selection_indices is None:
            with torch.no_grad():
                raw = (q.detach() @ k.detach().transpose(-2, -1)) * self.scale
                cm  = torch.triu(torch.full((N, N_total), float('-inf'), device=x.device), 1)
                imp = F.softmax(raw + cm[None, None], dim=-1).mean(1)   # (B,N,N_total)
                imp_m = imp.masked_fill(cm[None] == float('-inf'), -1e9)
                a_cap = min(budget, N_total)
                _, top_idx = torch.topk(imp_m, k=a_cap, dim=-1, sorted=False)
                self_idx  = (torch.arange(N_total - N, N_total, device=x.device)
                                 .view(1, -1, 1).expand(B, N, 1))
                selection_indices = torch.cat([top_idx, self_idx], dim=-1)  # (B,N,A+1)

            # ── Margin loss: recompute imp WITH grad (only if requested) ─
            if return_aux_loss:
                cm_g  = torch.triu(torch.full((N, N_total), float('-inf'), device=x.device), 1)
                imp_g = F.softmax((q @ k.transpose(-2,-1)) * self.scale + cm_g[None,None], dim=-1).mean(1)
                sel_s = torch.gather(imp_g, -1, selection_indices)
                min_s = sel_s.min(-1).values
                valid = cm_g[None].expand(B, N, N_total) != float('-inf')
                smask = torch.zeros((B, N, N_total), dtype=torch.bool, device=x.device)
                smask.scatter_(-1, selection_indices, True)
                non_s = imp_g.masked_fill(~(valid & ~smask), -1e9)
                mns   = non_s.max(-1).values
                mns   = torch.where(mns == -1e9, torch.zeros_like(mns), mns)
                aux_loss = F.relu(self.margin_delta - min_s + mns).mean()

        # ── Pass-2: efficient gather + FlashAttention-2 (sparse) ─────
        k_gath, v_gath = _gather_kv(k, v, selection_indices)
        out = _pass2_attention(q, k_gath, v_gath, self.scale)
        out = self.out_proj(out.transpose(1,2).contiguous().view(B, N, self.d_model))
        return out, selection_indices, aux_loss, new_kv


# ── Building blocks ──────────────────────────────────────────────────
class FFN(nn.Module):
    def __init__(self, d_model: int, ff_dim: Optional[int] = None, dropout: float = 0.0):
        super().__init__()
        ff_dim = ff_dim or d_model * 4
        self.net = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(ff_dim, d_model))
    def forward(self, x): return self.net(x)


class ASABlock(nn.Module):
    def __init__(self, d_model, nhead, max_a=64, is_router=True, dropout=0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = AdaptiveSelectiveAttention(d_model, nhead, max_a, is_router, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FFN(d_model, dropout=dropout)
    def forward(self, x, sel=None, max_a=None, ret_aux=False, kvc=None):
        a, sel, aux, kvc = self.attn(self.ln1(x), sel, max_a, ret_aux, kvc)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, sel, aux, kvc


class ASALayerGroup(nn.Module):
    """1 Router + (g-1) Followers sharing selection T."""
    def __init__(self, g, d_model, nhead, max_a=64, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            ASABlock(d_model, nhead, max_a, is_router=(i==0), dropout=dropout)
            for i in range(g)])
    def forward(self, x, max_a=None, ret_aux=False, g_kvc=None):
        shared_sel, aux_total, new_kvc = None, None, []
        for i, layer in enumerate(self.layers):
            x, sel, aux, kvc_new = layer(x, shared_sel, max_a, ret_aux, g_kvc[i] if g_kvc else None)
            if i == 0: shared_sel = sel
            if aux is not None: aux_total = aux if aux_total is None else aux_total + aux
            if kvc_new is not None: new_kvc.append(kvc_new)
        return x, aux_total, new_kvc


class ASATransformerGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=6,
                 max_seq_len=1024, group_size=2, max_a=64, dropout=0.0):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        groups, rem  = [], num_layers
        while rem > 0:
            g = min(group_size, rem); groups.append(ASALayerGroup(g, d_model, nhead, max_a, dropout)); rem -= g
        self.groups   = nn.ModuleList(groups)
        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size)

    def forward(self, x, max_a=None, return_aux_loss=False, kv_caches=None):
        B, N  = x.shape
        past  = kv_caches[0][0][0].shape[2] if (kv_caches and kv_caches[0]) else 0
        pos   = torch.arange(past, past + N, device=x.device).unsqueeze(0)
        h     = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        aux_t, new_kvc = None, []
        for i, grp in enumerate(self.groups):
            h, aux, gkvc = grp(h, max_a, return_aux_loss, kv_caches[i] if kv_caches else None)
            if aux is not None: aux_t = aux if aux_t is None else aux_t + aux
            if gkvc: new_kvc.append(gkvc)
        logits = self.lm_head(self.final_ln(h))
        return (logits, aux_t) if return_aux_loss else logits

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=60, max_a=None):
        self.eval(); dev = prompt.device; curr = prompt.clone()
        B, N = curr.shape
        h    = self.drop(self.tok_emb(curr) + self.pos_emb(torch.arange(N, device=dev).unsqueeze(0)))
        kvc  = []
        for grp in self.groups:
            h, _, gkvc = grp(h, max_a); kvc.append(gkvc)
        curr = torch.cat([curr, self.lm_head(self.final_ln(h))[:, -1, :].argmax(-1, keepdim=True)], 1)
        for _ in range(max_new_tokens - 1):
            if curr.size(1) >= self.max_seq_len: break
            pl  = curr.size(1) - 1
            h   = self.drop(self.tok_emb(curr[:,-1:]) + self.pos_emb(torch.tensor([[pl]], device=dev)))
            nkvc = []
            for i, grp in enumerate(self.groups):
                h, _, gkvc = grp(h, max_a, g_kvc=kvc[i]); nkvc.append(gkvc)
            kvc  = nkvc
            curr = torch.cat([curr, self.lm_head(self.final_ln(h))[:,-1,:].argmax(-1, keepdim=True)], 1)
        return curr


class DenseGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=6,
                 max_seq_len=1024, dropout=0.0):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        enc = nn.TransformerEncoderLayer(d_model, nhead, d_model*4, dropout,
                                          batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers, enable_nested_tensor=False)
        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, N = x.shape
        h    = self.tok_emb(x) + self.pos_emb(torch.arange(N, device=x.device).unsqueeze(0))
        mask = nn.Transformer.generate_square_subsequent_mask(N, device=x.device)
        return self.lm_head(self.final_ln(self.transformer(h, mask=mask, is_causal=True)))

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=60):
        self.eval(); curr = prompt.clone()
        for _ in range(max_new_tokens):
            if curr.size(1) >= self.max_seq_len: break
            # [:, -1, :] — integer index → (B, V) 2D, avoids 3D shape mismatch
            curr = torch.cat([curr, self.forward(curr)[:,-1,:].argmax(-1, keepdim=True)], 1)
        return curr


def count_params(m): return sum(p.numel() for p in m.parameters())
print('[OK] All model classes defined.')

In [ ]:
# ── Cell 4: Memory proof — gather index comparison ────────────────────
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    B, H, N_total, N, D, A_test = 4, 8, 1024, 1024, 64, 32
    k_t = torch.randn(B, H, N_total, D, device='cuda')
    v_t = torch.randn(B, H, N_total, D, device='cuda')
    sel = torch.randint(0, N_total, (B, N, A_test), device='cuda')
    kg, vg = _gather_kv(k_t, v_t, sel)
    torch.cuda.synchronize()
    peak    = torch.cuda.max_memory_allocated() / 1e6
    naive   = B * H * N * N_total * D * 4 / 1e6
    flat    = B * H * N * A_test * D * 8 / 1e6   # old flat_idx int64
    adv_idx = B * H * N * A_test * 8 / 1e6        # new sel_bh int64
    print(f'Naive expand+gather index   : {naive:.0f} MB')
    print(f'flat_idx approach           : {flat:.0f} MB (int64)')
    print(f'Advanced-index sel_bh (NEW) : {adv_idx:.1f} MB (int64)  ← {flat/adv_idx:.0f}x smaller')
    print(f'GPU peak actually used      : {peak:.0f} MB')
    print(f'Output shape: {kg.shape}  [PASS]')
    del k_t, v_t, sel, kg, vg; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
else:
    print('CPU mode — GPU proof skipped.')

In [ ]:
# ── Cell 5: Structured dependency dataset ────────────────────────────
VOCAB, SEQ_LEN = 16, 1024

def make_data(n=250, seq=SEQ_LEN):
    data  = np.random.choice([4,5,9,10,11], size=(n, seq)).astype(np.int64)
    rules = [(1,2),(6,7),(3,8)]
    for i in range(n):
        tok, dep = rules[i % 3]
        data[i,:4]=tok; data[i,250:255]=dep; data[i,750:755]=dep
    return torch.from_numpy(data[:,:-1]), torch.from_numpy(data[:,1:])

X_tr, Y_tr = make_data(200); X_te, Y_te = make_data(40)
print(f'Train: {X_tr.shape}   Test: {X_te.shape}   Vocab: {VOCAB}')

In [ ]:
# ── Cell 6: Build models ──────────────────────────────────────────────
D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP = 128, 4, 4, SEQ_LEN, 2

dense  = DenseGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ)
asa32  = ASATransformerGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP, max_a=32)
asa128 = ASATransformerGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP, max_a=128)

print(f'Dense GPT           : {count_params(dense):,} params')
print(f'ASA-GPT (max_a=32)  : {count_params(asa32):,} params')
print(f'ASA-GPT (max_a=128) : {count_params(asa128):,} params')

In [ ]:
# ── Cell 7: Training + benchmark utilities ────────────────────────────
def _sync():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
def reset_vram():
    if DEVICE.type == 'cuda': torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def peak_vram_mb():
    if DEVICE.type != 'cuda': return 0.0
    _sync(); return torch.cuda.max_memory_allocated() / 1e6

def train_model(model, X, Y, *, epochs=3, bs=8, lr=3e-3, is_asa=False, max_a=None):
    model.train().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        perm = torch.randperm(len(X)); loss_s, nb = 0.0, 0
        t0 = time.perf_counter()
        for i in range(0, len(X), bs):
            bx = X[perm[i:i+bs]].to(DEVICE); by = Y[perm[i:i+bs]].to(DEVICE)
            opt.zero_grad()
            lg = model(bx, max_a=max_a) if is_asa else model(bx)
            loss = F.cross_entropy(lg.view(-1, VOCAB), by.reshape(-1))
            loss.backward(); opt.step()
            loss_s += loss.item(); nb += 1
        print(f'  Epoch {ep+1}/{epochs} [{time.perf_counter()-t0:.2f}s]  loss={loss_s/nb:.4f}')

def eval_ppl(model, X, Y, *, is_asa=False, max_a=None, bs=8):
    model.eval(); total, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), bs):
            lg = model(X[i:i+bs].to(DEVICE), max_a=max_a) if is_asa else model(X[i:i+bs].to(DEVICE))
            total += F.cross_entropy(lg.view(-1,VOCAB), Y[i:i+bs].to(DEVICE).reshape(-1)).item(); n += 1
    return math.exp(total / n)

def bench_gen(model, *, is_asa=False, max_a=None, gen_tok=60, runs=6):
    model.eval(); p = torch.tensor([[1,1,1,1]], dtype=torch.int64, device=DEVICE)
    with torch.no_grad():
        if is_asa: model.generate(p, max_new_tokens=5, max_a=max_a)
        else:      model.generate(p, max_new_tokens=5)
    _sync(); times = []
    for _ in range(runs):
        _sync(); t0 = time.perf_counter()
        with torch.no_grad():
            if is_asa: model.generate(p, max_new_tokens=gen_tok, max_a=max_a)
            else:      model.generate(p, max_new_tokens=gen_tok)
        _sync(); times.append(time.perf_counter() - t0)
    mu, std = float(np.mean(times)), float(np.std(times))
    return mu, std, gen_tok / mu

print('[OK] Utilities ready.')

In [ ]:
# ── Cell 8: Standard Dense GPT ───────────────────────────────────────
print('='*60,'\n  Standard Dense GPT\n','='*60)
reset_vram()
train_model(dense, X_tr, Y_tr, epochs=3, bs=8, is_asa=False)
vtr_d = peak_vram_mb(); print(f'  VRAM (train): {vtr_d:.1f} MB')
ppl_d = eval_ppl(dense, X_te, Y_te, is_asa=False)
reset_vram()
t_d, s_d, tps_d = bench_gen(dense, is_asa=False, gen_tok=60)
vgen_d = peak_vram_mb()
print(f'  Perplexity  : {ppl_d:.4f}\n  Gen time    : {t_d:.4f}s ±{s_d:.5f}\n  Tok/sec     : {tps_d:.2f}\n  VRAM (gen)  : {vgen_d:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 9: ASA-GPT max_a=32 ─────────────────────────────────────────
print('='*60,'\n  ASA-GPT  max_a=32\n','='*60)
reset_vram()
train_model(asa32, X_tr, Y_tr, epochs=3, bs=8, is_asa=True, max_a=32)
vtr_32 = peak_vram_mb(); print(f'  VRAM (train): {vtr_32:.1f} MB')
ppl_32 = eval_ppl(asa32, X_te, Y_te, is_asa=True, max_a=32)
reset_vram()
t_32, s_32, tps_32 = bench_gen(asa32, is_asa=True, max_a=32, gen_tok=60)
vgen_32 = peak_vram_mb()
print(f'  Perplexity  : {ppl_32:.4f}\n  Gen time    : {t_32:.4f}s ±{s_32:.5f}\n  Tok/sec     : {tps_32:.2f}\n  VRAM (gen)  : {vgen_32:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 10: ASA-GPT max_a=128 ───────────────────────────────────────
print('='*60,'\n  ASA-GPT  max_a=128\n','='*60)
reset_vram()
train_model(asa128, X_tr, Y_tr, epochs=3, bs=8, is_asa=True, max_a=128)
vtr_128 = peak_vram_mb(); print(f'  VRAM (train): {vtr_128:.1f} MB')
ppl_128 = eval_ppl(asa128, X_te, Y_te, is_asa=True, max_a=128)
reset_vram()
t_128, s_128, tps_128 = bench_gen(asa128, is_asa=True, max_a=128, gen_tok=60)
vgen_128 = peak_vram_mb()
print(f'  Perplexity  : {ppl_128:.4f}\n  Gen time    : {t_128:.4f}s ±{s_128:.5f}\n  Tok/sec     : {tps_128:.2f}\n  VRAM (gen)  : {vgen_128:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 11: Final comparative table ─────────────────────────────────
rows = [
    ('Standard Dense GPT',  count_params(dense),  ppl_d,   vtr_d,   t_d,   s_d,   tps_d,   vgen_d),
    ('ASA-GPT (max_a=32)',  count_params(asa32),   ppl_32,  vtr_32,  t_32,  s_32,  tps_32,  vgen_32),
    ('ASA-GPT (max_a=128)', count_params(asa128),  ppl_128, vtr_128, t_128, s_128, tps_128, vgen_128),
]
W = 120
print('='*W)
print('  BENCHMARK — 1024 TOKENS / ~500K PARAMS / GPU T4 x2  (Pass-1 no_grad + adv-index + SDPA)')
print('='*W)
print(f"{'MODELO':<23} | {'PARAMS':>10} | {'PPL':>8} | {'VRAM-TR':>9} | {'GEN-T':>8} | {'STD':>7} | {'TOK/SEC':>9} | {'VRAM-GEN':>9}")
print('-'*W)
for name, p, ppl, vtr, t, std, tps, vgen in rows:
    print(f"{name:<23} | {p:>10,} | {ppl:>8.4f} | {vtr:>8.1f}M | {t:>8.4f}s | {std:>7.5f} | {tps:>9.2f} | {vgen:>8.1f}M")
print('='*W)

ref_tps, ref_vtr = rows[0][6], rows[0][3]
print('\nSpeedup tok/sec vs Dense GPT:')
for name, *_, tps, _ in rows: print(f'  {name:<23}: {tps/ref_tps:.3f}x')
print('\nVRAM-Training vs Dense GPT:')
for name, _, __, vtr, *_ in rows: print(f'  {name:<23}: {vtr:.1f} MB  ({(1-vtr/ref_vtr)*100:+.1f}%)')
print('\n[Done]')